# AION-C — Training Notebook (v2)

Entrena el modelo AION-C (Mixture-of-Specialists Encoder) en **5 fases** (0-4).

**Flujo rapido (smoke test en T4, < 5 min):**
1. Sube `aion_c.zip` al panel de archivos de Colab (o a Google Drive)
2. Ejecuta **Celda 1** (Setup) — auto-detecta datasets en `/content/` y `/content/drive/MyDrive/`
3. Ejecuta **Celda 2** (Smoke Test) — verifica que todo funciona antes de gastar GPU
4. Ejecuta **Celdas 3-7** en orden con `CONFIG = "tiny"`
5. Revisa el **Resumen** en Celda 8

**Entrenamiento real en H200 / Vast.ai:**
- Selecciona `CONFIG = "production"` en cada celda
- Monta Google Drive (`MOUNT_DRIVE = True`) para guardar checkpoints
- Phase 4 = Instruction Tuning con LoRA (28.5K ejemplos)

| Config | Modelo | Tiempo T4 (est.) | VRAM |
|--------|--------|-----------------|------|
| tiny | 64-dim, 2 layers | < 5 min | ~0.5 GB |
| medium | 768-dim, 6 layers | ~2-4 h | ~8 GB |
| production | 1024-dim, 14/24 layers | ~24-48 h | ~40+ GB (H200) |

**Fases:**
- Phase 0: Aprender a hablar (encoder + decoder)
- Phase 1: Backbone compartido (pipeline completo)
- Phase 2: Motores especializados (CORA, FORGE-C, AXIOM, MUSE, EMPATHY)
- Phase 3: Fine-tune E2E + Orchestrator routing
- Phase 4: Instruction Tuning con LoRA rank=16

---

In [ ]:
# @title 1. Setup: instalar dependencias y preparar entorno
# @markdown ### Instrucciones
# @markdown 1. Sube **`aion_c.zip`** al panel de archivos de Colab o a Google Drive
# @markdown    - Para datos reales sube tambien **`DataSet-Generator-Claude-Opus.zip`**
# @markdown 2. Ajusta los parametros de abajo si necesario
# @markdown 3. Ejecuta esta celda **antes** de las demas

import subprocess, sys, os, zipfile, glob
from pathlib import Path

# --- Parametros ---
ZIP_PATH     = ''                     # @param {type:"string"}
EXTRACT_TO   = '/content'             # @param {type:"string"}
DATASET_ROOT = ''                     # @param {type:"string"}
MOUNT_DRIVE  = False                  # @param {type:"boolean"}
# ZIP_PATH: vacio = auto-buscar en /content/ y /content/drive/MyDrive/
# DATASET_ROOT: vacio = auto-detectar

print('=' * 60)
print('  AION-C Training Notebook v2 -- Setup')
print('=' * 60)

# -- 1. Google Drive (opcional) --
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('[OK] Google Drive montado')
    except Exception as e:
        print(f'[WARN] No se pudo montar Drive: {e}')

# -- 2. Instalar sentencepiece --
print('\n[1/5] Instalando dependencias...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece'],
    capture_output=True, text=True)
if r.returncode != 0:
    print(f'  WARN: {r.stderr[:300]}')
print('  OK sentencepiece')

# -- 3. Buscar y descomprimir ZIP --
print(f'\n[2/5] Buscando zip...')
_zip_candidates = []
if ZIP_PATH:
    _zip_candidates.append(Path(ZIP_PATH))
# Buscar en /content/ y en Drive
for _base in ['/content', '/content/drive/MyDrive', '/content/drive/MyDrive/Colab Notebooks']:
    _bp = Path(_base)
    if _bp.exists():
        for name in ['aion_c.zip', 'AION-C.zip', 'ias.zip']:
            _zip_candidates.append(_bp / name)

zip_p = None
for _c in _zip_candidates:
    if _c.exists():
        zip_p = _c
        break

if zip_p is not None:
    print(f'  Encontrado: {zip_p}')
    with zipfile.ZipFile(str(zip_p), 'r') as zf:
        zf.extractall(EXTRACT_TO)
    print(f'  OK extraido en {EXTRACT_TO}')
else:
    print('  No se encontro zip. Buscando AION-C ya extraido...')

# Si existe un zip de datasets, extraerlo tambien
for _base in [Path(EXTRACT_TO), Path('/content/drive/MyDrive')]:
    _ds_zip = _base / 'DataSet-Generator-Claude-Opus.zip'
    if _ds_zip.exists():
        print(f'  Encontrado {_ds_zip}, extrayendo...')
        with zipfile.ZipFile(str(_ds_zip), 'r') as zf:
            zf.extractall(EXTRACT_TO)
        print('  OK DataSet-Generator-Claude-Opus extraido')
        break

# -- 4. Localizar AION-C --
print('\n[3/5] Localizando AION-C...')
AION_ROOT = None
_search_bases = [
    Path(EXTRACT_TO),
    Path('/content/drive/MyDrive'),
    Path('/content/drive/MyDrive/Colab Notebooks'),
]
for _base in _search_bases:
    for _sub in ['AION-C', 'ias/AION-C', '.']:
        _c = _base / _sub
        if (_c / 'router' / 'pipeline.py').exists():
            AION_ROOT = _c
            break
    if AION_ROOT:
        break

# Fallback: rglob
if AION_ROOT is None:
    for _base in _search_bases:
        if _base.exists():
            for _p in _base.rglob('pipeline.py'):
                if _p.parent.name == 'router':
                    AION_ROOT = _p.parent.parent
                    break
        if AION_ROOT:
            break

if AION_ROOT is None:
    raise RuntimeError(
        'No se encontro AION-C.\n'
        'Sube aion_c.zip al panel de archivos o a Google Drive.')

print(f'  OK AION-C en: {AION_ROOT}')
os.chdir(str(AION_ROOT))
if str(AION_ROOT) not in sys.path:
    sys.path.insert(0, str(AION_ROOT))

# -- 5. Localizar datasets Opus --
print('\n[4/5] Localizando datasets Opus...')
_ds_sub = 'mose_distillation_datasets/datasets'
_ds_cands = []
if DATASET_ROOT:
    _ds_cands.append(Path(DATASET_ROOT))
for _base in [AION_ROOT.parent, Path(EXTRACT_TO), Path('/content/drive/MyDrive')]:
    _ds_cands.append(_base / 'DataSet-Generator-Claude-Opus' / _ds_sub)
    _ds_cands.append(_base / 'ias' / 'DataSet-Generator-Claude-Opus' / _ds_sub)

_DATASET_ROOT = None
for _c in _ds_cands:
    if _c.exists() and (_c / 'mose_cora.jsonl').exists():
        _DATASET_ROOT = _c
        break

if _DATASET_ROOT:
    print(f'  OK datasets en: {_DATASET_ROOT}')
else:
    print('  WARN: datasets Opus no encontrados -- fallback sintetico activo')

# -- 6. Verificar instruction tuning dataset --
print('\n[5/5] Verificando instruction tuning dataset...')
_it_path = AION_ROOT / 'datasets' / 'instruction_tuning.jsonl'
if _it_path.exists():
    _it_lines = sum(1 for _ in open(str(_it_path), 'r', encoding='utf-8'))
    print(f'  OK instruction_tuning.jsonl: {_it_lines:,} ejemplos')
else:
    print('  Generando instruction tuning dataset...')
    from synth.instruction_gen import InstructionGenerator, write_jsonl
    _gen = InstructionGenerator(seed=42)
    _examples = _gen.generate_all()
    write_jsonl(_examples, _it_path)
    print(f'  OK generado: {len(_examples):,} ejemplos')

# -- Resumen --
import torch
print()
print('=' * 60)
_gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
_vram = ''
if torch.cuda.is_available():
    _vram = f' ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)'
print(f'  AION-C root  : {AION_ROOT}')
print(f'  Datasets Opus: {_DATASET_ROOT or "NO (fallback sintetico)"}')
print(f'  IT dataset   : {_it_path}')
print(f'  GPU          : {_gpu}{_vram}')
print('=' * 60)
print('\nSetup completo. Ejecuta Celda 2 (Smoke Test) para verificar.')

In [ ]:
# @title 2. Smoke Test -- Verificar que todo funciona antes de entrenar
# @markdown Ejecuta el smoke test completo en CPU con config=tiny.
# @markdown **Los 13 tests deben pasar antes de gastar en GPU.**
# @markdown Si algun test falla, el output dice exactamente cual.

import subprocess, sys, os

# Asegurar encoding UTF-8 para los prints del smoke test
os.environ['PYTHONIOENCODING'] = 'utf-8'

print('=' * 60)
print('  SMOKE TEST -- Verificando todas las piezas')
print('=' * 60)
print()

result = subprocess.run(
    [sys.executable, '-m', 'experiments.smoke_test_full'],
    capture_output=False,
    text=True,
    timeout=600,
    env={**os.environ, 'PYTHONIOENCODING': 'utf-8'},
)

if result.returncode == 0:
    print('\n*** SMOKE TEST PASSED -- Safe to proceed with training ***')
else:
    print('\n*** SMOKE TEST FAILED -- Fix issues before training ***')

In [ ]:
# @title 3. Phase 0: Aprender a Hablar (Encoder + Decoder)
# @markdown **Phase 0**: entrena solo encoder y decoder (sin motores).
# @markdown `graph_repr = zeros` bypasea los motores.
# @markdown
# @markdown **Presets de steps:** tiny=300, medium=3000, production=20000

CONFIG    = 'tiny'   # @param ["tiny", "medium", "production"]
MAX_STEPS = 0        # @param {type:"integer"}
RESUME    = False    # @param {type:"boolean"}

import copy, sys, torch
from pathlib import Path

for _c in ['/content/AION-C', '/content/ias/AION-C']:
    if Path(_c + '/router/pipeline.py').exists():
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

from experiments.train_production import (
    build_pipeline_and_tok, load_all_datasets, run_phase0, _PRESETS)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Run dir: Drive if available, else /content/runs/
_drive = Path('/content/drive/MyDrive')
RUN_DIR = (_drive / 'aion_runs' / f'aion_{CONFIG}') if _drive.exists() \
          else Path(f'/content/runs/aion_{CONFIG}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'Run dir: {RUN_DIR}')

# Build pipeline (reutiliza si ya existe con el mismo config)
if not ('_ph_pipeline' in dir() and _ph_config == CONFIG):
    print(f'\nConstruyendo pipeline (config={CONFIG})...')
    _ph_pipeline, _ph_tok, _ph_cfg = build_pipeline_and_tok(CONFIG, device)
    _ph_config   = CONFIG
    _ph_hparams  = copy.copy(_PRESETS[CONFIG])
    # dataset_root explicito: usa _DATASET_ROOT de la celda de Setup
    _ds_root = _DATASET_ROOT if '_DATASET_ROOT' in dir() else None
    _ph_datasets = load_all_datasets(
        max_examples=_ph_hparams.n_train + _ph_hparams.n_val + 100,
        eval_size=_ph_hparams.n_val,
        dataset_root=_ds_root)
    _ph_results  = {}
    print(f'Params      : {sum(p.numel() for p in _ph_pipeline.parameters()):,}')
    print(f'Dominios    : {list(_ph_datasets.keys())}')
    print(f'Dataset root: {_ds_root or "auto (fallback sintetico)"}')
else:
    print(f'Reutilizando pipeline (config={_ph_config})')

hp = copy.copy(_ph_hparams)
if MAX_STEPS > 0:
    hp.ph0_steps = MAX_STEPS
    hp.ph0_eval  = max(5, MAX_STEPS // 4)

print(f'\nSteps: {hp.ph0_steps}  eval_every: {hp.ph0_eval}  '
      f'patience: {hp.ph0_patience}  lr: {hp.ph0_lr:.1e}')

_ph_results['phase0'] = run_phase0(
    pipeline=_ph_pipeline, cfg=_ph_cfg, tok=_ph_tok,
    datasets=_ph_datasets, hparams=hp, device=device,
    checkpoint_dir=RUN_DIR, resume=RESUME,
    max_steps_override=MAX_STEPS if MAX_STEPS > 0 else None)

print('\n' + '=' * 60)
print('Phase 0 completada:')
print(_ph_results['phase0'].summary_line())
if device.type == 'cuda':
    print(f'VRAM usada: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# @title 4. Phase 1: Backbone Compartido (Pipeline Completo)
# @markdown **Phase 1**: entrena el pipeline completo con datos mezclados.
# @markdown Carga checkpoint de Phase 0 si existe.
# @markdown
# @markdown **Presets de steps:** tiny=300, medium=3000, production=30000

CONFIG    = 'tiny'   # @param ["tiny", "medium", "production"]
MAX_STEPS = 0        # @param {type:"integer"}
RESUME    = False    # @param {type:"boolean"}

import copy, sys, torch
from pathlib import Path

for _c in ['/content/AION-C', '/content/ias/AION-C']:
    if Path(_c + '/router/pipeline.py').exists():
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

from experiments.train_production import (
    build_pipeline_and_tok, load_all_datasets, run_phase1, _PRESETS)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

_drive = Path('/content/drive/MyDrive')
RUN_DIR = (_drive / 'aion_runs' / f'aion_{CONFIG}') if _drive.exists() \
          else Path(f'/content/runs/aion_{CONFIG}')
RUN_DIR.mkdir(parents=True, exist_ok=True)

if not ('_ph_pipeline' in dir() and _ph_config == CONFIG):
    print(f'\nConstruyendo pipeline (config={CONFIG})...')
    _ph_pipeline, _ph_tok, _ph_cfg = build_pipeline_and_tok(CONFIG, device)
    _ph_config   = CONFIG
    _ph_hparams  = copy.copy(_PRESETS[CONFIG])
    _ds_root = _DATASET_ROOT if '_DATASET_ROOT' in dir() else None
    _ph_datasets = load_all_datasets(
        max_examples=_ph_hparams.n_train + _ph_hparams.n_val + 100,
        eval_size=_ph_hparams.n_val,
        dataset_root=_ds_root)
    _ph_results  = {}
else:
    print(f'Reutilizando pipeline (config={_ph_config})')

hp = copy.copy(_ph_hparams)
if MAX_STEPS > 0:
    hp.ph1_steps = MAX_STEPS
    hp.ph1_eval  = max(5, MAX_STEPS // 4)

print(f'Steps: {hp.ph1_steps}  lr: {hp.ph1_lr:.1e}')

_ph_results['phase1'] = run_phase1(
    pipeline=_ph_pipeline, cfg=_ph_cfg, tok=_ph_tok,
    datasets=_ph_datasets, hparams=hp, device=device,
    checkpoint_dir=RUN_DIR, resume=RESUME,
    max_steps_override=MAX_STEPS if MAX_STEPS > 0 else None)

print('\n' + '=' * 60)
print('Phase 1 completada:')
print(_ph_results['phase1'].summary_line())
if device.type == 'cuda':
    print(f'VRAM usada: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# @title 5. Phase 2: Motores Especializados (por dominio)
# @markdown **Phase 2**: entrena cada motor con su dataset especifico.
# @markdown Backbone unfrozen para grad flow, pero solo motor params en optimizer.
# @markdown
# @markdown **Presets:** tiny=200/motor, medium=2000/motor, production=20000/motor

CONFIG    = 'tiny'   # @param ["tiny", "medium", "production"]
MOTORS    = 'all'    # @param ["all", "cora", "forge_c", "axiom", "muse", "empathy"]
MAX_STEPS = 0        # @param {type:"integer"}
RESUME    = False    # @param {type:"boolean"}

import copy, sys, torch
from pathlib import Path

for _c in ['/content/AION-C', '/content/ias/AION-C']:
    if Path(_c + '/router/pipeline.py').exists():
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

from experiments.train_production import (
    build_pipeline_and_tok, load_all_datasets, run_phase2, _PRESETS)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

_drive = Path('/content/drive/MyDrive')
RUN_DIR = (_drive / 'aion_runs' / f'aion_{CONFIG}') if _drive.exists() \
          else Path(f'/content/runs/aion_{CONFIG}')
RUN_DIR.mkdir(parents=True, exist_ok=True)

if not ('_ph_pipeline' in dir() and _ph_config == CONFIG):
    print(f'\nConstruyendo pipeline (config={CONFIG})...')
    _ph_pipeline, _ph_tok, _ph_cfg = build_pipeline_and_tok(CONFIG, device)
    _ph_config   = CONFIG
    _ph_hparams  = copy.copy(_PRESETS[CONFIG])
    _ds_root = _DATASET_ROOT if '_DATASET_ROOT' in dir() else None
    _ph_datasets = load_all_datasets(
        max_examples=_ph_hparams.n_train + _ph_hparams.n_val + 100,
        eval_size=_ph_hparams.n_val,
        dataset_root=_ds_root)
    _ph_results  = {}
else:
    print(f'Reutilizando pipeline (config={_ph_config})')

motors_list = None if MOTORS == 'all' else [MOTORS]
hp = copy.copy(_ph_hparams)
if MAX_STEPS > 0:
    hp.ph2_steps = MAX_STEPS
    hp.ph2_eval  = max(5, MAX_STEPS // 4)

print(f'Motores: {motors_list or "todos"}  Steps/motor: {hp.ph2_steps}  lr: {hp.ph2_lr:.1e}')

_ph2_list = run_phase2(
    pipeline=_ph_pipeline, cfg=_ph_cfg, tok=_ph_tok,
    datasets=_ph_datasets, hparams=hp, device=device,
    checkpoint_dir=RUN_DIR, resume=RESUME,
    max_steps_override=MAX_STEPS if MAX_STEPS > 0 else None,
    motors=motors_list)
_ph_results['phase2'] = _ph2_list

print('\n' + '=' * 60)
print('Phase 2 completada:')
for r in _ph2_list:
    print(r.summary_line())
if device.type == 'cuda':
    print(f'VRAM usada: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# @title 6. Phase 3: Fine-tune E2E + Orchestrator Routing
# @markdown **Phase 3a**: entrena el orchestrator para routing correcto.
# @markdown **Phase 3b**: fine-tune end-to-end del pipeline completo con lr bajo.
# @markdown
# @markdown **Presets:** tiny=100+150 steps, medium=1000+2000, production=10000+20000

CONFIG    = 'tiny'   # @param ["tiny", "medium", "production"]
MAX_STEPS = 0        # @param {type:"integer"}
RESUME    = False    # @param {type:"boolean"}

import copy, sys, torch
from pathlib import Path

for _c in ['/content/AION-C', '/content/ias/AION-C']:
    if Path(_c + '/router/pipeline.py').exists():
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

from experiments.train_production import (
    build_pipeline_and_tok, load_all_datasets, run_phase3, _PRESETS)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

_drive = Path('/content/drive/MyDrive')
RUN_DIR = (_drive / 'aion_runs' / f'aion_{CONFIG}') if _drive.exists() \
          else Path(f'/content/runs/aion_{CONFIG}')
RUN_DIR.mkdir(parents=True, exist_ok=True)

if not ('_ph_pipeline' in dir() and _ph_config == CONFIG):
    print(f'\nConstruyendo pipeline (config={CONFIG})...')
    _ph_pipeline, _ph_tok, _ph_cfg = build_pipeline_and_tok(CONFIG, device)
    _ph_config   = CONFIG
    _ph_hparams  = copy.copy(_PRESETS[CONFIG])
    _ds_root = _DATASET_ROOT if '_DATASET_ROOT' in dir() else None
    _ph_datasets = load_all_datasets(
        max_examples=_ph_hparams.n_train + _ph_hparams.n_val + 100,
        eval_size=_ph_hparams.n_val,
        dataset_root=_ds_root)
    _ph_results  = {}
else:
    print(f'Reutilizando pipeline (config={_ph_config})')

hp = copy.copy(_ph_hparams)
if MAX_STEPS > 0:
    hp.ph3_orch_steps = MAX_STEPS
    hp.ph3_e2e_steps  = MAX_STEPS
    hp.ph3_eval       = max(5, MAX_STEPS // 4)

print(f'Orchestrator: {hp.ph3_orch_steps} steps  E2E: {hp.ph3_e2e_steps} steps')

_ph_results['phase3'] = run_phase3(
    pipeline=_ph_pipeline, cfg=_ph_cfg, tok=_ph_tok,
    datasets=_ph_datasets, hparams=hp, device=device,
    checkpoint_dir=RUN_DIR, resume=RESUME,
    max_steps_override=MAX_STEPS if MAX_STEPS > 0 else None)

print('\n' + '=' * 60)
print('Phase 3 completada:')
print(_ph_results['phase3'].summary_line())
if device.type == 'cuda':
    print(f'VRAM usada: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# @title 7. Phase 4: Instruction Tuning con LoRA
# @markdown **Phase 4**: aplica LoRA rank=16 al decoder y entrena con 28.5K ejemplos
# @markdown de instruction tuning (identidad, codigo, math, creatividad, MEM, etc.)
# @markdown
# @markdown Carga checkpoint de Phase 3 si existe. Solo LoRA params son entrenables (~85K).

CONFIG      = 'tiny'   # @param ["tiny", "medium", "production"]
MAX_STEPS   = 0        # @param {type:"integer"}
INTERACTIVE = False    # @param {type:"boolean"}
# INTERACTIVE=True abre un prompt cada eval_every steps para probar el modelo

import copy, sys, torch
from pathlib import Path

for _c in ['/content/AION-C', '/content/ias/AION-C']:
    if Path(_c + '/router/pipeline.py').exists():
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

from experiments.train_production import (
    build_pipeline_and_tok, load_all_datasets, run_phase4, _PRESETS)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

_drive = Path('/content/drive/MyDrive')
RUN_DIR = (_drive / 'aion_runs' / f'aion_{CONFIG}') if _drive.exists() \
          else Path(f'/content/runs/aion_{CONFIG}')
RUN_DIR.mkdir(parents=True, exist_ok=True)

if not ('_ph_pipeline' in dir() and _ph_config == CONFIG):
    print(f'\nConstruyendo pipeline (config={CONFIG})...')
    _ph_pipeline, _ph_tok, _ph_cfg = build_pipeline_and_tok(CONFIG, device)
    _ph_config   = CONFIG
    _ph_hparams  = copy.copy(_PRESETS[CONFIG])
    _ph_results  = {}
else:
    print(f'Reutilizando pipeline (config={_ph_config})')

hp = copy.copy(_ph_hparams)
if MAX_STEPS > 0:
    hp.ph4_steps = MAX_STEPS
    hp.ph4_eval  = max(5, MAX_STEPS // 4)

print(f'Steps: {hp.ph4_steps}  LoRA rank: 16  Interactive: {INTERACTIVE}')

_ph_results['phase4'] = run_phase4(
    pipeline=_ph_pipeline, cfg=_ph_cfg, tok=_ph_tok,
    hparams=hp, device=device,
    checkpoint_dir=RUN_DIR,
    max_steps_override=MAX_STEPS if MAX_STEPS > 0 else None,
    interactive=INTERACTIVE)

print('\n' + '=' * 60)
print('Phase 4 completada:')
print(_ph_results['phase4'].summary_line())
if device.type == 'cuda':
    print(f'VRAM usada: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# @title 8. Resumen Final
# @markdown Resultados de todas las fases ejecutadas en esta sesion.

import json, sys
from pathlib import Path

for _c in ['/content/AION-C', '/content/ias/AION-C']:
    if Path(_c + '/router/pipeline.py').exists():
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

if '_ph_results' not in dir() or not _ph_results:
    print('No hay resultados en memoria. Ejecuta las celdas 3-7 primero.')
else:
    print('=' * 65)
    print('  RESUMEN FINAL -- Sesion actual')
    print('=' * 65)
    total_s = 0.0
    for key in ('phase0', 'phase1', 'phase3', 'phase4'):
        if key in _ph_results:
            r = _ph_results[key]
            print(r.summary_line())
            total_s += r.elapsed_s
    if 'phase2' in _ph_results:
        for r in _ph_results['phase2']:
            print(r.summary_line())
            total_s += r.elapsed_s
    print()
    print(f'  Tiempo total sesion: {total_s/60:.1f} min')
    print('=' * 65)

# -- Checkpoints en disco --
_config = _ph_config if '_ph_config' in dir() else 'tiny'
_drive  = Path('/content/drive/MyDrive')
_rd = (_drive / 'aion_runs' / f'aion_{_config}') if _drive.exists() \
     else Path(f'/content/runs/aion_{_config}')

print(f'\nCheckpoints en: {_rd}')
if _rd.exists():
    ckpts = sorted(_rd.glob('*.pt'))
    if ckpts:
        for ck in ckpts:
            size_mb = ck.stat().st_size / 1e6
            print(f'  {ck.name:<40}  {size_mb:>7.1f} MB')
    else:
        print('  (ninguno todavia)')

# -- VRAM --
import torch
if torch.cuda.is_available():
    print(f'\nVRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
    print(f'VRAM peak     : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')
    print(f'VRAM reserved : {torch.cuda.memory_reserved() / 1e9:.2f} GB')

# -- Estado del modelo --
if '_ph_pipeline' in dir():
    total_p = sum(p.numel() for p in _ph_pipeline.parameters())
    final_ck = _rd / 'phase4_instruction.pt'
    if not final_ck.exists():
        final_ck = _rd / 'phase3_final.pt'
    print(f'\nModelo en memoria:')
    print(f'  config         : {_config}')
    print(f'  params totales : {total_p:,}  ({total_p/1e6:.1f}M)')
    print(f'  device         : {next(_ph_pipeline.parameters()).device}')
    if final_ck.exists():
        size_mb = final_ck.stat().st_size / 1e6
        print(f'  checkpoint     : {final_ck.name}  ({size_mb:.1f} MB)')

In [ ]:
# @title 9. Test de Escalado (medium) -- Medir VRAM real
# @markdown Construye el pipeline `medium` (768-dim, 6 layers, ~50M params) y mide
# @markdown VRAM real con un forward pass + backward. Util para estimar si cabe en tu GPU.
# @markdown
# @markdown **No entrena** -- solo mide memoria. Safe to run.

import sys, torch, gc
from pathlib import Path

for _c in ['/content/AION-C', '/content/ias/AION-C']:
    if Path(_c + '/router/pipeline.py').exists():
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

from experiments.train_production import build_pipeline_and_tok

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type != 'cuda':
    print('Esta celda requiere GPU. En CPU no hay VRAM que medir.')
    print('Puedes ejecutarla en Colab con GPU habilitada.')
else:
    # Limpiar VRAM
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    print('=' * 60)
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('=' * 60)

    vram_before = torch.cuda.memory_allocated()

    print('\n[1/4] Construyendo pipeline medium...')
    _test_pipe, _test_tok, _test_cfg = build_pipeline_and_tok('medium', device)
    vram_model = torch.cuda.memory_allocated()
    print(f'  VRAM modelo: {(vram_model - vram_before) / 1e9:.3f} GB')

    print('\n[2/4] Forward pass (seq_len=512, batch=1)...')
    ids = torch.randint(1, _test_cfg.vocab_size, (1, 512), device=device)
    with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
        out = _test_pipe(ids)
    vram_fwd = torch.cuda.memory_allocated()
    print(f'  VRAM forward: {(vram_fwd - vram_model) / 1e9:.3f} GB')

    print('\n[3/4] Backward pass...')
    loss = out.logits.sum()
    loss.backward()
    vram_bwd = torch.cuda.memory_allocated()
    print(f'  VRAM backward: {(vram_bwd - vram_fwd) / 1e9:.3f} GB')

    print('\n[4/4] Resumen VRAM:')
    print(f'  Modelo           : {(vram_model - vram_before) / 1e9:.3f} GB')
    print(f'  + Activaciones   : {(vram_fwd - vram_model) / 1e9:.3f} GB')
    print(f'  + Gradientes     : {(vram_bwd - vram_fwd) / 1e9:.3f} GB')
    print(f'  = Total allocated: {vram_bwd / 1e9:.3f} GB')
    print(f'  Peak allocated   : {torch.cuda.max_memory_allocated() / 1e9:.3f} GB')
    print(f'  Peak reserved    : {torch.cuda.max_memory_reserved() / 1e9:.3f} GB')

    total_vram = torch.cuda.get_device_properties(0).total_memory
    headroom = (total_vram - torch.cuda.max_memory_allocated()) / 1e9
    print(f'  Headroom         : {headroom:.3f} GB')

    if headroom < 1.0:
        print('\n  WARNING: Headroom < 1 GB. Consider reducing batch_size or max_seq_len.')
    else:
        print(f'\n  OK: {headroom:.1f} GB de margen. Safe for training.')

    # Cleanup
    del _test_pipe, _test_tok, _test_cfg, out, loss, ids
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\n  After cleanup: {torch.cuda.memory_allocated() / 1e9:.3f} GB')

In [ ]:
# @title 10. Probar el Modelo Manualmente
# @markdown Escribe un texto y ve la respuesta del modelo en su estado actual.
# @markdown Funciona despues de cualquier fase (incluso Phase 0).

PROMPT = "Quien eres?"  # @param {type:"string"}

import sys, torch
from pathlib import Path

for _c in ['/content/AION-C', '/content/ias/AION-C']:
    if Path(_c + '/router/pipeline.py').exists():
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

if '_ph_pipeline' not in dir():
    print('ERROR: No hay pipeline en memoria. Ejecuta al menos la celda de Phase 0.')
else:
    from experiments.train_production import _encode
    device = next(_ph_pipeline.parameters()).device

    _ph_pipeline.eval()
    ids = _encode(_ph_tok, PROMPT, 128)
    ids_t = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        out = _ph_pipeline(ids_t)

    pred_ids = out.logits[0].argmax(dim=-1).tolist()
    try:
        response = _ph_tok.decode(pred_ids)
    except Exception:
        response = str(pred_ids[:30])

    print(f'Input   : {PROMPT}')
    print(f'Response: {response}')
    print(f'Confidence: {out.confidence.tolist() if out.confidence is not None else "N/A"}')